# Step 2 — Catalogue cleaning and duplicate review

**Project:** Skincare product catalogue · **Input:** `moisturizers_with_cosing.csv` · **Status:** provisional until manual checks are complete.

> **Validated against the uploaded source CSV (1 November 2026).** The execution confirmed **244 rows × 17 source columns**, one category (`moisturizer`), **0 fully identical raw rows**, **0 repeated non-null codes**, and **8 similar-name candidate pairs**. Three product names have whitespace differences. **No product has been removed.**
>
> Ingredient list quality remains an open issue. The parser flags **11 rows**: 2 known non-lists, 6 pre-identified irregular/OCR records, and 3 one-entry lists that may be valid single-ingredient products. A one-entry list is **not automatically a parsing error**.

## Purpose
Standardize product categories, brands and names; identify exact and near-duplicate records; inspect inconsistent ingredient-list formatting; and create **reviewable interim outputs** without changing the original CSV.

**Important rules**
- Keep `data/raw/` unchanged; do not infer that matching names or ingredient sets prove identical products.
- Keep barcodes as **strings** (leading zeroes matter) and preserve `source_row` for traceability.
- Never treat an unparseable ingredient list as an empty formulation. Missing/unverified entries remain flagged.
- Ingredient sorting is **for comparison only**; the original declared ingredient order remains in `ingredients_text`.
- This notebook **does not automatically delete** any candidate duplicate. A reviewed, documented merge should be a separate deliberate action.

### Workflow
1. Setup and source snapshot → 2. Catalogue inspection → 3. Category/name normalization → 4. Exact/code/name duplicate detection → 5. Ingredient-format inspection → 6. Pairwise ingredient comparison → 7. Manual review export/import → 8. Provisional export and QA.

## 2.1 — Setup and load a protected working copy
Run cells **from top to bottom**. Install optional packages in the notebook kernel if needed: `%pip install pandas rapidfuzz openpyxl`.

In [1]:
import pandas as pd
import re
from pathlib import Path
from itertools import combinations
from rapidfuzz import fuzz

project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
raw_path = project_dir / "data" / "raw"
interim_path = project_dir / "data" / "interim"
interim_path.mkdir(parents=True, exist_ok=True)
input_file = raw_path / "moisturizers_with_cosing.csv"
assert input_file.exists(), f"Cannot locate source: {input_file}"

df = pd.read_csv(input_file, dtype={"code": "string"})
df_clean = df.copy()
df_clean["source_row"] = range(len(df_clean))
print("Source records:", len(df), "| Source columns:", len(df.columns))
print("Path:", input_file)

Source records: 244 | Source columns: 17
Path: /Users/linanawwas/code/imanemoujahid00-oss/Swan/skincare_project/data/raw/moisturizers_with_cosing.csv


## 2.2 — Baseline inspection
Check the real fields and missing data before edits. Counts in this notebook are calculated from your file, not hardcoded assumptions.

In [2]:
display(df_clean.head(5))
print("Columns:", df.columns.tolist())
print("Missing critical fields:")
display(df_clean[["code", "brands", "product_name", "ingredients_text"]].isna().sum().to_frame("missing_count"))
print("Original data types:")
df.info()

,role,code,product_name,brands,france,eu,ingredients_text,product_row_id,parsed_ingredient_count,matched_ingredient_count,ambiguous_ingredient_count,unmatched_ingredient_count,match_fraction,cosing_ids,matched_inci_names,cosing_functions,unmatched_ingredients,source_row
0,moisturizer,0860015354624,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories,False,False,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,190,1,0,0,1,0.0000,NaN,NaN,NaN,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,0
1,moisturizer,0860015354648,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories,False,False,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,191,1,0,0,1,0.0000,NaN,NaN,NaN,Water. Glycerin. Copper Tripeptide-1. Phenoxye...,1
2,moisturizer,4070765004748,Gesichtscreme Niacinamide LSF 30,Balea,False,False,"AQUA, NIACINAMIDE, DIBUTYL ADIPATE, GLYCERIN, ...",192,25,25,0,0,1.0000,31713 | 31959 | 32858 | 33893 | 33895 | 33896 ...,AQUA | BIS-ETHYLHEXYLOXYPHENOL METHOXYPHENYL T...,ANTIMICROBIAL | ANTISTATIC | BLEACHING | BUFFE...,NaN,2
3,moisturizer,8437018443420,RF Facial Cream,Masderm,False,True,"Aqua, Paraffinum Liquidum, Glyceryl Stearate, ...",193,33,30,0,3,0.9091,31959 | 32858 | 33401 | 34040 | 34103 | 34924 ...,AQUA | ASTROCARYUM MURUMURU SEED BUTTER | BIDE...,ANTICORROSIVE | ANTIFOAMING | ANTIMICROBIAL | ...,Citrus Aurantium Bergamia Oil | Tetramethyl Ac...,3
4,moisturizer,7891010255725,Hydro Boost,Neutrogena,False,False,"Aqua/água purificada, Dimethicone/dimeticona, ...",194,19,0,0,19,0.0000,NaN,NaN,NaN,Aqua/água purificada | Dimethicone/dimeticona ...,4


Columns: ['role', 'code', 'product_name', 'brands', 'france', 'eu', 'ingredients_text', 'product_row_id', 'parsed_ingredient_count', 'matched_ingredient_count', 'ambiguous_ingredient_count', 'unmatched_ingredient_count', 'match_fraction', 'cosing_ids', 'matched_inci_names', 'cosing_functions', 'unmatched_ingredients']
Missing critical fields:


,missing_count
code,0
brands,2
product_name,0
ingredients_text,0


Original data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   role                        244 non-null    object 
 1   code                        244 non-null    string 
 2   product_name                244 non-null    object 
 3   brands                      242 non-null    object 
 4   france                      244 non-null    bool   
 5   eu                          244 non-null    bool   
 6   ingredients_text            244 non-null    object 
 7   product_row_id              244 non-null    int64  
 8   parsed_ingredient_count     244 non-null    int64  
 9   matched_ingredient_count    244 non-null    int64  
 10  ambiguous_ingredient_count  244 non-null    int64  
 11  unmatched_ingredient_count  244 non-null    int64  
 12  match_fraction              244 non-null    float64
 13  cosing_ids    

## 2.3 — Standardize the category label
The previously observed `role` value was `moisturizer` for all 244 records. Recheck here; this validates **label consistency**, not whether every product is truly a moisturizer.

In [3]:
print("Original role distribution:")
display(df_clean["role"].value_counts(dropna=False))
df_clean["category_clean"] = df_clean["role"].astype("string").str.strip().str.casefold()
category_counts = df_clean["category_clean"].value_counts(dropna=False).rename_axis("category").reset_index(name="product_count")
display(category_counts)
category_counts.to_csv(interim_path / "category_counts.csv", index=False)

Original role distribution:


role
moisturizer    244
Name: count, dtype: int64

,category,product_count
0,moisturizer,244


## 2.4 — Standardize names and create matching keys
Keep the original text. Remove extra whitespace in **display columns** and use case-insensitive keys only for matching. Do not remove sizes/SPF/version terms, because they may distinguish real products.

In [4]:
for source, target in [("brands", "brand_clean"), ("product_name", "product_name_clean")]:
    df_clean[target] = df_clean[source].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
df_clean["brand_key"] = df_clean["brand_clean"].fillna("").str.casefold()
df_clean["product_key"] = df_clean["product_name_clean"].fillna("").str.casefold()
for original, cleaned in [("brands", "brand_clean"), ("product_name", "product_name_clean")]:
    changed = (df_clean[original].astype("string") != df_clean[cleaned]).fillna(False).sum()
    print(f"{original} spacing changes: {changed}")
display(df_clean[["brands", "brand_clean", "product_name", "product_name_clean"]].head(10))
df_clean.to_csv(interim_path / "catalogue_standardized.csv", index=False)

brands spacing changes: 0
product_name spacing changes: 3


,brands,brand_clean,product_name,product_name_clean
0,PLU Laboratories,PLU Laboratories,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories GHK-Cu Face Tonic
1,PLU Laboratories,PLU Laboratories,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories GHK-Cu Face Tonic
2,Balea,Balea,Gesichtscreme Niacinamide LSF 30,Gesichtscreme Niacinamide LSF 30
3,Masderm,Masderm,RF Facial Cream,RF Facial Cream
4,Neutrogena,Neutrogena,Hydro Boost,Hydro Boost
5,Byoma,Byoma,Hydrating recovery oil,Hydrating recovery oil
6,Stanhome,Stanhome,All Purpose Crema Multiuso Profumo di Fico,All Purpose Crema Multiuso Profumo di Fico
7,Byphasse,Byphasse,Crème ultra-hydratante,Crème ultra-hydratante
8,Diadermine,Diadermine,ΕΝΥΔΑΤΙΚΗ ΚΡΕΜΑ ΣΕ ΜΟΡΦΗ ΤΖΕΛ,ΕΝΥΔΑΤΙΚΗ ΚΡΕΜΑ ΣΕ ΜΟΡΦΗ ΤΖΕΛ
9,Ceelini,Ceelini,Ceelini Yoğun Nemlendirici Vücut ve Yüz Kremi ...,Ceelini Yoğun Nemlendirici Vücut ve Yüz Kremi ...


## 2.5 — Detect exact duplicates and repeated product codes
**Exact duplicates** are identical rows in the original dataframe; `source_row` would make every row unique, so use `df`. Repeated barcodes are **review candidates**, not automatic deletions.

In [5]:
exact_duplicates = df[df.duplicated(keep=False)].copy()
repeated_codes = df_clean[df_clean["code"].notna() & df_clean["code"].duplicated(keep=False)].copy()
print("Exact duplicate rows (all occurrences):", len(exact_duplicates))
print("Repeated-code records:", len(repeated_codes), "| distinct repeated codes:", repeated_codes["code"].nunique())
display(repeated_codes[["source_row", "code", "brands", "product_name"]].sort_values("code"))
exact_duplicates.to_csv(interim_path / "exact_duplicates_review.csv", index=False)
repeated_codes.to_csv(interim_path / "duplicate_codes_review.csv", index=False)

Exact duplicate rows (all occurrences): 0
Repeated-code records: 0 | distinct repeated codes: 0


,source_row,code,brands,product_name


## 2.6 — Detect similar product names
Use RapidFuzz within the same known brand. The threshold of **90/100** produces review candidates; it is not a duplicate decision. Missing brands need separate attention because brand grouping skips them.

In [6]:
candidates = []
for brand, group in df_clean.groupby("brand_key"):
    if not brand:
        continue
    for i, j in combinations(group.index, 2):
        a, b = df_clean.loc[i, "product_key"], df_clean.loc[j, "product_key"]
        if not a or not b:
            continue
        score = fuzz.ratio(a, b)
        if score >= 90:
            candidates.append({
                "row_1": int(df_clean.loc[i, "source_row"]), "row_2": int(df_clean.loc[j, "source_row"]),
                "brand": brand, "product_1": df_clean.loc[i, "product_name"],
                "product_2": df_clean.loc[j, "product_name"], "name_similarity": round(score, 2)
            })
possible_duplicates_df = pd.DataFrame(candidates, columns=["row_1", "row_2", "brand", "product_1", "product_2", "name_similarity"])
print("Similar-name candidate pairs:", len(possible_duplicates_df))
display(possible_duplicates_df)
possible_duplicates_df.to_csv(interim_path / "possible_duplicates.csv", index=False)

Similar-name candidate pairs: 8


,row_1,row_2,brand,product_1,product_2,name_similarity
0,175,176,babylove,leichte Gesichtscreme,leichte Gesichtscreme,100.00
1,49,56,bioderma,Bioderma Atoderm Cream Ultra,Bioderma Atoderm Creme Ultra,96.43
2,85,136,bioderma,Sébium sensitive,Sébium sensitive,100.00
3,78,191,kiehl's,Ultra Facial Cream,Ultra facial cream,100.00
4,108,186,l'oréal,Revitalift filler,Revitalift filler,100.00
5,32,84,nivea,Nivea creme,Nivea Creme,100.00
6,0,1,plu laboratories,PLU Laboratories GHK-Cu Face Tonic,PLU Laboratories GHK-Cu Face Tonic,100.00
7,62,63,the inkey list,"Vitamin B, C and E moisturizer","Vitamin B, C and E moisturizer",100.00


## 2.7 — Inspect and compare ingredient lists

### Data-quality finding
The original notebook used comma splitting, but some source records use **bullets**, **newlines**, **periods**, or OCR-like text. For instance, PLU Laboratories rows 0/1 are period-separated; L’Oréal row 108 has ambiguous space-separated names; rows 6 and 9 contain *descriptions, not ingredient lists*.

**Conservative policy:** Automatically split only high-confidence delimiters (commas, bullets and newlines; a short, clearly period-separated list is handled only for the known PLU rows). Other records must be **flagged and externally verified**. No generic splitting on periods or spaces. Preserve original strings and ordering.

**Limitations:** Lowercasing/sorting does not reconcile INCI synonyms (`water` vs `aqua`), and set similarity ignores order and concentrations. Full INCI alias normalization belongs to Step 4.

In [7]:
def parse_ingredients(text, source_row):
    """Return comparison-only alphabetized terms and a conservative review flag."""
    if pd.isna(text) or not str(text).strip():
        return [], "missing_ingredient_list"
    raw = str(text).strip()
    if raw.casefold() in {"ingredienti non visibili nell'immagine.", "günlük nemlendirici krem"}:
        return [], "missing_ingredient_list"
    raw = re.sub(r"\[/\]\(https://file\+\.vscode-resource\.vscode-cdn\.net/\)", " / ", raw)
    raw = re.sub(r"[•·⚫]", ",", raw)
    raw = re.sub(r"\r?\n+", ",", raw)
    # Rows 0 and 1 have been inspected: period-delimited six-item PLU lists.
    if source_row in (0, 1):
        raw = re.sub(r"\.\s+", ", ", raw)
    items = [s.strip().casefold().strip(" .;") for s in raw.split(",") if s.strip(" .;")]
    status = "parsed_candidate" if len(items) >= 2 else "needs_manual_review"
    # Known irregular/OCR or space-separated source entries require further verification,
    # even when the generic parser produces more than one token.
    if source_row in (39, 95, 99, 108, 114, 190):
        status = "needs_manual_review"
    return sorted(items), status

parsed = [parse_ingredients(t, int(r)) for t, r in zip(df_clean["ingredients_text"], df_clean["source_row"])]
df_clean["ingredients_sorted"] = [x[0] for x in parsed]
df_clean["ingredient_review_status"] = [x[1] for x in parsed]
summary = df_clean["ingredient_review_status"].value_counts(dropna=False)
display(summary.to_frame("records"))
display(df_clean[df_clean["ingredient_review_status"] != "parsed_candidate"][["source_row", "brands", "product_name", "ingredient_review_status"]])

,records
ingredient_review_status,
parsed_candidate,233
needs_manual_review,9
missing_ingredient_list,2


,source_row,brands,product_name,ingredient_review_status
6,6,Stanhome,All Purpose Crema Multiuso Profumo di Fico,missing_ingredient_list
9,9,Ceelini,Ceelini Yoğun Nemlendirici Vücut ve Yüz Kremi ...,missing_ingredient_list
39,39,Garnier,Hautklar,needs_manual_review
95,95,CHANEL,Micro crème/ HYDRA BEAUTY,needs_manual_review
99,99,La Roche-Posay,Nutric Intense Rich In-depth Nutri-Reconstitut...,needs_manual_review
100,100,Mentholatum Aust. Pty Ltd,Oxy 5,needs_manual_review
108,108,L'Oréal,Revitalift filler,needs_manual_review
114,114,Jonzac,Eau thermale Jonzac Pure soin purifiant anti-i...,needs_manual_review
146,146,Zenzitude,huile de jojoba vierge,needs_manual_review
190,190,Yves Rocher,Crème fondamentale Anti-âge global,needs_manual_review


### Inspect flagged products and original ingredient texts
The parser is a triage aid. Treat `parsed_candidate` as **not independently verified**, especially when OCR artifacts or aliases could be present.

In [8]:
flagged = df_clean[df_clean["ingredient_review_status"] != "parsed_candidate"].copy()
flagged[["source_row", "code", "brands", "product_name", "ingredients_text", "ingredient_review_status"]].to_csv(
    interim_path / "ingredient_parsing_review.csv", index=False)
for row in [0, 1, 6, 9, 108, 186]:
    p = df_clean[df_clean["source_row"] == row].iloc[0]
    print(f"Row {row} | {p['product_name']} | status={p['ingredient_review_status']} | parsed entries={len(p['ingredients_sorted'])}")
    print("Original:", p["ingredients_text"])
    print("Parsed:", p["ingredients_sorted"], "\n")

Row 0 | PLU Laboratories GHK-Cu Face Tonic | status=parsed_candidate | parsed entries=6
Original: Water. Glycerin. Copper Tripeptide-1. Phenoxyethanol. Palmitoyl Tripeptide-1. Palmitoyl Tetrapeptide-7.
Parsed: ['copper tripeptide-1', 'glycerin', 'palmitoyl tetrapeptide-7', 'palmitoyl tripeptide-1', 'phenoxyethanol', 'water'] 

Row 1 | PLU Laboratories GHK-Cu Face Tonic | status=parsed_candidate | parsed entries=6
Original: Water. Glycerin. Copper Tripeptide-1. Phenoxyethanol. Palmitoyl Tripeptide-1. Palmitoyl Tetrapeptide-7.
Parsed: ['copper tripeptide-1', 'glycerin', 'palmitoyl tetrapeptide-7', 'palmitoyl tripeptide-1', 'phenoxyethanol', 'water'] 

Row 6 | All Purpose Crema Multiuso Profumo di Fico | status=missing_ingredient_list | parsed entries=0
Original: Ingredienti non visibili nell'immagine.
Parsed: [] 

Row 9 | Ceelini Yoğun Nemlendirici Vücut ve Yüz Kremi Aloe Vera,Alpha Arbutin ve  Hyalorunic Acid Özlü 50 ML | status=missing_ingredient_list | parsed entries=0
Original: Günlü

### Pairwise ingredient overlap
The metric below is **Jaccard similarity** of distinct lowercased strings: `100 × |A∩B| / |A∪B|`. It is **missing** when either list cannot be trusted, not automatically 0%. Results are exploratory and should not decide merges.

In [9]:
comparison = []
for _, cand in possible_duplicates_df.iterrows():
    i, j = int(cand["row_1"]), int(cand["row_2"])
    a = df_clean.loc[df_clean["source_row"] == i].iloc[0]
    b = df_clean.loc[df_clean["source_row"] == j].iloc[0]
    set_a, set_b = set(a["ingredients_sorted"]), set(b["ingredients_sorted"])
    usable = (a["ingredient_review_status"] == "parsed_candidate" and b["ingredient_review_status"] == "parsed_candidate" and bool(set_a) and bool(set_b))
    comparison.append({"row_1": i, "row_2": j, "brand": cand["brand"],
        "ingredients_1": len(set_a), "ingredients_2": len(set_b),
        "common_ingredients": len(set_a & set_b) if usable else pd.NA,
        "ingredient_similarity_pct": round(100 * len(set_a & set_b) / len(set_a | set_b), 2) if usable else pd.NA,
        "comparison_status": "provisional_comparison" if usable else "unverified_ingredient_list",
        "only_row_1": "; ".join(sorted(set_a - set_b)) if usable else "",
        "only_row_2": "; ".join(sorted(set_b - set_a)) if usable else ""})
ingredient_comparison = pd.DataFrame(comparison)
display(ingredient_comparison)
ingredient_comparison.to_csv(interim_path / "ingredient_comparison.csv", index=False)

,row_1,row_2,brand,ingredients_1,ingredients_2,common_ingredients,ingredient_similarity_pct,comparison_status,only_row_1,only_row_2
0,175,176,babylove,22,22,18,69.23,provisional_comparison,caprylyl glycol; ethylhexylglycerin; sodium st...,lactylate; p-anisic acid; sodium polyacrylate;...
1,49,56,bioderma,25,25,25,100.0,provisional_comparison,,
2,85,136,bioderma,25,25,25,100.0,provisional_comparison,,
3,78,191,kiehl's,35,34,18,35.29,provisional_comparison,acrylates / c10-30 alkyl acrylate crosspolymer...,acrylates/c10-30 alkyl acrylate crosspolymer; ...
4,108,186,l'oréal,2,13,<NA>,<NA>,unverified_ingredient_list,,
5,32,84,nivea,15,21,13,56.52,provisional_comparison,lanolin alcohol (eucerit®); sodium anisate,benzyl benzoate; cinnamyl alcohol; citronellol...
6,0,1,plu laboratories,6,6,6,100.0,provisional_comparison,,
7,62,63,the inkey list,23,24,22,88.0,provisional_comparison,cocos nucifera (coconut) oil,(coconut) oil; cocos nucifera


## 2.8 — Duplicate decision log and manual corrections

**Decision rules:** `merge` only after verifying that two records represent the same catalogue entity and selecting a record to retain; `keep_separate` for distinct package/formulation entries; `needs_review` otherwise. Review each barcode, package size, formulation/version, manufacturer label, and source URL.

**File safety:** The next cell *creates a review template only if it does not already exist*, so subsequent notebook runs will not overwrite your Excel/CSV edits. Review notes from the original work should be treated as hypotheses, not verified decisions.

In [10]:
duplicate_review_path = interim_path / "duplicate_manual_review.csv"
if not duplicate_review_path.exists():
    reviews = []
    for _, p in possible_duplicates_df.iterrows():
        a = df_clean.loc[df_clean["source_row"] == int(p["row_1"])].iloc[0]
        b = df_clean.loc[df_clean["source_row"] == int(p["row_2"])].iloc[0]
        reviews.append({**p.to_dict(), "code_1": a["code"], "code_2": b["code"],
                        "same_code": bool(pd.notna(a["code"]) and pd.notna(b["code"]) and a["code"] == b["code"]),
                        "review_decision": "needs_review", "review_notes": "", "verification_source": ""})
    pd.DataFrame(reviews).to_csv(duplicate_review_path, index=False)
    print("Created review template:", duplicate_review_path)
else:
    print("Preserved existing review file:", duplicate_review_path)
review_df = pd.read_csv(duplicate_review_path, dtype={"code_1": "string", "code_2": "string"})
display(review_df.head())

Preserved existing review file: /Users/linanawwas/code/imanemoujahid00-oss/Swan/skincare_project/data/interim/duplicate_manual_review.csv


,row_1,row_2,brand,product_1,product_2,similarity,code_1,code_2,same_code,same_ingredients,review_decision,review_notes,verification_source
0,175,176,babylove,leichte Gesichtscreme,leichte Gesichtscreme,100.000000,4058172066153,4010355141873,False,False,keep_separate,Both records have identical product names but ...,https://incidecoder.com/products/dm-babylove-b...
1,49,56,bioderma,Bioderma Atoderm Cream Ultra,Bioderma Atoderm Creme Ultra,96.428571,8699956514185,8699956513676,False,False,keep_separate,Same Bioderma Atoderm Cream Ultra product fami...,https://www.ahona.store/products/bioderma-atod...
2,85,136,bioderma,Sébium sensitive,Sébium sensitive,100.000000,3401360106994,06010699,False,True,needs_review,Both records have identical names and all 25 i...,https://www.pharmalite.com/prodotto/sebium-sen...
3,78,191,kiehl's,Ultra Facial Cream,Ultra facial cream,100.000000,3605970360757,3605971545672,False,False,keep_separate,Both records belong to the Kiehl's Ultra Facia...,https://www.kiehls.com/Ultra-Facial-Cream/3605...
4,108,186,l'oréal,Revitalift filler,Revitalift filler,100.000000,3600523201501,3600523732548,False,False,keep_separate,Barcode 3600523201501 identifies L'Oreal Paris...,https://superpharm.it/loreal-paris-revitalift-...


### Manual ingredient review workbook
Create an `.xlsx` template for the **eight initially selected manual-review rows** (6, 9, 39, 95, 99, 108, 114, 190). Three more one-entry records (100, 146, 226) are flagged for source verification and exported to a **separate review file**. They may be genuine one-ingredient products. The initial eight-row template is not exhaustive. If a reviewed file already exists, it will **not** be overwritten. To use a researched workbook, copy it into `data/interim` and verify its source and approval status.

In [11]:
manual_review_rows = [6, 9, 39, 95, 99, 108, 114, 190]
manual_review_path = interim_path / "manual_ingredient_review.xlsx"
if not manual_review_path.exists():
    manual_review = df_clean[df_clean["source_row"].isin(manual_review_rows)][
        ["source_row", "code", "brands", "product_name", "ingredients_text", "ingredient_review_status"]].copy()
    manual_review["corrected_ingredients"] = ""
    manual_review["verification_source"] = ""
    manual_review["review_decision"] = "pending"
    manual_review["review_notes"] = ""
    manual_review.to_excel(manual_review_path, index=False)
    print("Created:", manual_review_path)
else:
    print("Preserved existing:", manual_review_path)
reviewed_ingredients = pd.read_excel(manual_review_path, dtype={"code": "string"})
display(reviewed_ingredients[["source_row", "product_name", "review_decision"]])

Preserved existing: /Users/linanawwas/code/imanemoujahid00-oss/Swan/skincare_project/data/interim/manual_ingredient_review.xlsx


,source_row,product_name,review_decision
0,6,All Purpose Crema Multiuso Profumo di Fico,unavailable
1,9,Ceelini Yoğun Nemlendirici Vücut ve Yüz Kremi ...,verified_retailer
2,39,Hautklar,verified_retailer
3,95,Micro crème/ HYDRA BEAUTY,verified_retailer
4,99,Nutric Intense Rich In-depth Nutri-Reconstitut...,verified_manufacturer
5,108,Revitalift filler,verified_retailer
6,114,Eau thermale Jonzac Pure soin purifiant anti-i...,verified_retailer
7,190,Crème fondamentale Anti-âge global,verified_secondary_source


### Additional single-ingredient verification candidates

Rows **100 (Oxy 5), 146 (Zenzitude jojoba oil), and 226 (Melvita jojoba oil)** each have one ingredient entry. This might be correct; confirm against the product label or brand record. These are **separate from the eight previously prioritized manual-review rows**. The code below writes an additional review file without modifying existing review decisions.

In [12]:
additional_rows = [100, 146, 226]
additional_review_path = interim_path / "additional_single_ingredient_review.csv"
if not additional_review_path.exists():
    additional_review = df_clean[df_clean["source_row"].isin(additional_rows)][
        ["source_row", "code", "brands", "product_name", "ingredients_text", "ingredient_review_status"]
    ].copy()
    additional_review["review_decision"] = "pending"
    additional_review["verification_source"] = ""
    additional_review["review_notes"] = "Check if one-ingredient label is accurate"
    additional_review.to_csv(additional_review_path, index=False)
    print("Created:", additional_review_path)
else:
    print("Preserved existing:", additional_review_path)


Preserved existing: /Users/linanawwas/code/imanemoujahid00-oss/Swan/skincare_project/data/interim/additional_single_ingredient_review.csv


## 2.9 — Provisional catalogue and quality checks

**Current policy:** Keep all products while duplicate review is unresolved. This dataset is **standardized / provisionally cleaned**, not fully deduplicated. Do not automatically apply external ingredient corrections unless their `review_decision` and sources have been checked. This notebook therefore exports the flagged status but preserves raw ingredient text.

**Remaining work before finalizing Step 2:** Resolve eight candidate pairs and any repeated-code groups; verify ambiguous ingredients (especially L’Oréal row 108); inspect category anomalies; record keep/merge decisions and evidence. Then produce the final deduplicated export with a documented row-to-row merge log.

In [13]:
catalogue_cleaned = df_clean.copy()
assert len(catalogue_cleaned) == len(df)
assert catalogue_cleaned["source_row"].is_unique
assert set(catalogue_cleaned["source_row"]) == set(range(len(df)))
output_file = interim_path / "catalogue_provisional.csv"
catalogue_cleaned.to_csv(output_file, index=False)
print("Saved provisional catalogue:", output_file)
print("Records retained:", len(catalogue_cleaned))
print("Missing ingredients (known non-lists):", (catalogue_cleaned["ingredient_review_status"] == "missing_ingredient_list").sum())
print("Manual ingredient review needed:", (catalogue_cleaned["ingredient_review_status"] == "needs_manual_review").sum())

Saved provisional catalogue: /Users/linanawwas/code/imanemoujahid00-oss/Swan/skincare_project/data/interim/catalogue_provisional.csv
Records retained: 244
Missing ingredients (known non-lists): 2
Manual ingredient review needed: 9


In [14]:
cleaning_summary = pd.DataFrame([{
    "source_records": len(df),
    "provisional_records": len(catalogue_cleaned),
    "original_columns": len(df.columns),
    "unique_clean_categories": df_clean["category_clean"].nunique(dropna=True),
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "repeated_code_records": len(repeated_codes),
    "similar_name_candidate_pairs": len(possible_duplicates_df),
    "unresolved_review_decisions": int((review_df["review_decision"] == "needs_review").sum()),
    "ingredient_records_flagged": int((df_clean["ingredient_review_status"] != "parsed_candidate").sum()),
    "confirmed_removed_rows": 0,
    "status": "provisional; manual verification pending"
}])
cleaning_summary.to_csv(interim_path / "cleaning_summary.csv", index=False)
display(cleaning_summary)

,source_records,provisional_records,original_columns,unique_clean_categories,exact_duplicate_rows,repeated_code_records,similar_name_candidate_pairs,unresolved_review_decisions,ingredient_records_flagged,confirmed_removed_rows,status
0,244,244,17,1,0,0,8,2,11,0,provisional; manual verification pending


## Deliverables and handoff

| File in `data/interim/` | Meaning |
|---|---|
| `category_counts.csv` | Distribution of normalized categories |
| `catalogue_standardized.csv` | Standardized product names and category |
| `exact_duplicates_review.csv` | All occurrences of identical raw rows |
| `duplicate_codes_review.csv` | Repeated barcode candidates |
| `possible_duplicates.csv` | Name-similarity candidates (RapidFuzz ≥90) |
| `ingredient_parsing_review.csv` | Records with flagged ingredient parsing |
| `ingredient_comparison.csv` | Provisional pairwise Jaccard comparison |
| `duplicate_manual_review.csv` | Persisted decisions and evidence (never auto-overwritten) |
| `manual_ingredient_review.xlsx` | Ingredient correction and source review template |
| `catalogue_provisional.csv` | Working standardized catalogue, all source rows retained |
| `cleaning_summary.csv` | Reproducible counts and unresolved work |

### Methodological notes
- Name similarity and ingredients overlap are *screening indicators*, not proof of product identity.
- Identical ingredient sets may still differ in concentration, order, size or formulation version.
- Data-quality status must not be confused with a regulatory or scientific assessment of the products.
- A later frozen `catalogue_v1` should have a version manifest, validation checks, and a completed decision log.


### Validated findings on the uploaded snapshot
- **244 source records; 17 original fields**; all `role=moisturizer`.
- **No exact duplicated original rows** and **no repeated non-null product codes**.
- **Eight name-similar candidate pairs**, all requiring product-identity decisions; do not merge based only on ingredient overlap.
- **Eleven ingredient-review flags** in this conservative parser (including three potentially valid single-ingredient entries).
- Jaccard similarities from this parser can differ from older notebook results because formatting and tokenization changed. The L'Oréal pair remains **unverified**, not 0% similar.
- **244 records retained** in the provisional export. No ingredient list has been overwritten with unverified external text.


## Manual Review 

In [15]:
import pandas as pd

review_path = (
    interim_path / "manual_ingredient_review.xlsx"
)

ingredient_review = pd.read_excel(review_path)

display(ingredient_review)

,source_row,code,brands,product_name,ingredients_text,ingredient_review_status,corrected_ingredients,verification_source,review_decision,review_notes
0,6,64260729521,Stanhome,All Purpose Crema Multiuso Profumo di Fico,Ingredienti non visibili nell'immagine.,missing_ingredient_list,NaN,https://www.stanhome.it/products/all-purpose-3...,unavailable,Original ingredient list is unavailable. A sim...
1,9,8684149131485,Ceelini,Ceelini Yoğun Nemlendirici Vücut ve Yüz Kremi ...,Günlük nemlendirici krem,missing_ingredient_list,"Aqua, Glycerin, Hyaluronic Acid, Hypericum Per...",https://www.baby.bg/product/ceelini-ezhedneven...,verified_retailer,11 ingredients retrieved from a retailer listi...
2,39,4084200772901,Garnier,Hautklar,603258 115 - INGREDIENTS: AQUA / WATER • COCO-...,needs_manual_review,"Aqua / Water, Coco-Betaine, Propylene Glycol, ...",https://www.rewe.de/shop/p/garnier-hautklar-an...,verified_retailer,Ingredient list matches retailer formulation r...
3,95,3145891410709,CHANEL,Micro crème/ HYDRA BEAUTY,• • AQUA (WATER) GLYCERIN ALCOHOL PROPANEDIOL ...,needs_manual_review,"Aqua (Water), Glycerin, Alcohol, Propanediol, ...",https://beiqonic.com/chanel-hydra-beauty-micro...,verified_retailer,38 ingredients matched against a retailer list...
4,99,3337872413575,La Roche-Posay,Nutric Intense Rich In-depth Nutri-Reconstitut...,AQUA/WATER GLYCERIN • CETEARYL ALCOHOL CAPRYLI...,needs_manual_review,"Aqua / Water, Glycerin, Cetearyl Alcohol, Capr...",https://www.laroche-posay.ca/en/nutritic-inten...,verified_manufacturer,22 ingredient entries verified against the off...
5,108,3600523201501,L'Oréal,Revitalift filler,Aqua/ water. GLYCERIN DIMETHICONE ISOHEXADECAN...,needs_manual_review,"Aqua / Water, Glycerin, Dimethicone, Isohexade...",https://drogerienatura.pl/produkt/loreal-paris...,verified_retailer,40 ingredient entries transcribed from a retai...
6,114,3517360020793,Jonzac,Eau thermale Jonzac Pure soin purifiant anti-i...,AQUA/WATER/EAU • HAMAMELIS VIRGINIANA (WITCH H...,needs_manual_review,"Aqua / Water / Eau, Hamamelis Virginiana (Witc...",https://www.pharmaciedesdrakkars.com/jonzac-pu...,verified_retailer,Ingredient list transcribed from a pharmacy li...
7,190,3660005148038,Yves Rocher,Crème fondamentale Anti-âge global,AQUA (EAU) GLYCERIN COCO-CAPRYLATE/CAPRATE MET...,needs_manual_review,"Aqua, Glycerin, Coco-Caprylate/Caprate, Methyl...",https://www.codecheck.info/p/kosmetik/gesichts...,verified_secondary_source,Historical ingredient list closely matches the...


In [16]:
print("Total records:", len(ingredient_review))

print("\nReview decisions:")
print(
    ingredient_review["review_decision"]
    .value_counts(dropna=False)
)

print("\nProducts requiring review:")

display(
    ingredient_review[
        [
            "source_row",
            "brands",
            "product_name",
            "ingredient_review_status",
            "review_decision"
        ]
    ]
)

Total records: 8

Review decisions:
review_decision
verified_retailer            5
unavailable                  1
verified_manufacturer        1
verified_secondary_source    1
Name: count, dtype: int64

Products requiring review:


,source_row,brands,product_name,ingredient_review_status,review_decision
0,6,Stanhome,All Purpose Crema Multiuso Profumo di Fico,missing_ingredient_list,unavailable
1,9,Ceelini,Ceelini Yoğun Nemlendirici Vücut ve Yüz Kremi ...,missing_ingredient_list,verified_retailer
2,39,Garnier,Hautklar,needs_manual_review,verified_retailer
3,95,CHANEL,Micro crème/ HYDRA BEAUTY,needs_manual_review,verified_retailer
4,99,La Roche-Posay,Nutric Intense Rich In-depth Nutri-Reconstitut...,needs_manual_review,verified_manufacturer
5,108,L'Oréal,Revitalift filler,needs_manual_review,verified_retailer
6,114,Jonzac,Eau thermale Jonzac Pure soin purifiant anti-i...,needs_manual_review,verified_retailer
7,190,Yves Rocher,Crème fondamentale Anti-âge global,needs_manual_review,verified_secondary_source


In [17]:
# Select Stanhome row 6
mask = ingredient_review["source_row"] == 6

# Record the review result
ingredient_review.loc[
    mask, "review_decision"
] = "unavailable"

ingredient_review.loc[
    mask, "corrected_ingredients"
] = ""

ingredient_review.loc[
    mask, "verification_source"
] = "https://www.stanhome.it/products/all-purpose-300-ml"

# Add a note explaining why
ingredient_review.loc[
    mask, "review_notes"
] = (
    "Original ingredient list is unavailable. "
    "A similar official Stanhome product was found, "
    "but the exact barcode and limited-edition "
    "formulation could not be confirmed."
)

display(
    ingredient_review.loc[
        mask,
        [
            "source_row",
            "product_name",
            "review_decision",
            "review_notes"
        ]
    ]
)

,source_row,product_name,review_decision,review_notes
0,6,All Purpose Crema Multiuso Profumo di Fico,unavailable,Original ingredient list is unavailable. A sim...


In [18]:
ingredient_review.to_excel(
    interim_path / "manual_ingredient_review.xlsx",
    index=False
)

print("Stanhome review saved successfully!")

Stanhome review saved successfully!


In [19]:
# Select row 9
mask = ingredient_review["source_row"] == 9

ingredients = [
    "Aqua",
    "Glycerin",
    "Hyaluronic Acid",
    "Hypericum Perforatum Oil",
    "Ascorbic Acid",
    "Folic Acid",
    "Cetearyl Alcohol",
    "Aloe Barbadensis Leaf Extract",
    "Alpha-Arbutin",
    "Phenoxyethanol",
    "Ethylhexylglycerin"
]

ingredient_review.loc[
    mask, "corrected_ingredients"
] = ", ".join(ingredients)

ingredient_review.loc[
    mask, "verification_source"
] = "https://www.baby.bg/product/ceelini-ezhedneven-hidratirasht-krem-za-litse-50-ml/"

ingredient_review.loc[
    mask, "review_decision"
] = "verified_retailer"

ingredient_review.loc[
    mask, "review_notes"
] = (
    "11 ingredients retrieved from a retailer listing "
    "matching barcode 8684149131485. "
    "Manufacturer-label confirmation remains pending."
)

display(
    ingredient_review.loc[
        mask,
        [
            "source_row",
            "corrected_ingredients",
            "review_decision",
            "review_notes"
        ]
    ]
)

,source_row,corrected_ingredients,review_decision,review_notes
1,9,"Aqua, Glycerin, Hyaluronic Acid, Hypericum Per...",verified_retailer,11 ingredients retrieved from a retailer listi...


In [20]:
ingredient_review.to_excel(
    interim_path / "manual_ingredient_review.xlsx",
    index=False
)

print("Ceelini review saved successfully!")

Ceelini review saved successfully!


In [21]:
# Select Garnier row 39
mask = ingredient_review["source_row"] == 39

ingredients = [
    "Aqua / Water",
    "Coco-Betaine",
    "Propylene Glycol",
    "Sodium Laureth Sulfate",
    "PEG-120 Methyl Glucose Dioleate",
    "Disodium Cocoamphodiacetate",
    "Sodium Chloride",
    "Benzyl Alcohol",
    "Benzyl Salicylate",
    "Eucalyptus Globulus Leaf Extract",
    "Limonene",
    "Linalool",
    "Menthol",
    "Menthoxypropanediol",
    "Salicylic Acid",
    "Sodium Benzoate",
    "Tetrasodium EDTA",
    "Zinc Gluconate",
    "Parfum / Fragrance"
]

ingredient_review.loc[
    mask, "corrected_ingredients"
] = ", ".join(ingredients)

ingredient_review.loc[
    mask, "verification_source"
] = "https://www.rewe.de/shop/p/garnier-hautklar-anti-pickel-waschgel-150ml/705870"

ingredient_review.loc[
    mask, "review_decision"
] = "verified_retailer"

ingredient_review.loc[
    mask, "review_notes"
] = (
    "Ingredient list matches retailer formulation "
    "reference B183571/1. "
    "Corrected bullet separators and removed metadata. "
    "Product is a facial cleanser, not a moisturizer. "
    "Category correction required."
)

display(
    ingredient_review.loc[
        mask,
        [
            "source_row",
            "product_name",
            "review_decision",
            "review_notes"
        ]
    ]
)

,source_row,product_name,review_decision,review_notes
2,39,Hautklar,verified_retailer,Ingredient list matches retailer formulation r...


In [22]:
if "category_review_decision" not in df_clean.columns:
    df_clean["category_review_decision"] = ""

df_clean.loc[
    df_clean["source_row"] == 39,
    "category_review_decision"
] = "reclassify_as_cleanser"

display(
    df_clean.loc[
        df_clean["source_row"] == 39,
        [
            "source_row",
            "product_name",
            "role",
            "category_review_decision"
        ]
    ]
)

,source_row,product_name,role,category_review_decision
39,39,Hautklar,moisturizer,reclassify_as_cleanser


In [23]:
ingredient_review.to_excel(
    interim_path / "manual_ingredient_review.xlsx",
    index=False
)

print("Garnier review saved successfully!")

Garnier review saved successfully!


In [24]:
# Select CHANEL row 95
mask = ingredient_review["source_row"] == 95

ingredients = [
    "Aqua (Water)",
    "Glycerin",
    "Alcohol",
    "Propanediol",
    "Butylene Glycol",
    "Isodecyl Neopentanoate",
    "Isononyl Isononanoate",
    "Pentylene Glycol",
    "Isododecane",
    "Jojoba Esters",
    "Camellia Japonica Flower Extract",
    "Zingiber Officinale (Ginger) Root Extract",
    "Butyrospermum Parkii (Shea) Butter",
    "Methyl Methacrylate Crosspolymer",
    "Nylon-6",
    "Phenoxyethanol",
    "Dextrin Palmitate",
    "Neopentyl Glycol Diheptanoate",
    "Squalane",
    "Hydrogenated Polydecene",
    "Sodium Acrylates/C10-30 Alkyl Acrylate Crosspolymer",
    "Parfum (Fragrance)",
    "Sodium Carbomer",
    "Dipropylene Glycol",
    "Hydroxystearic Acid",
    "Sodium Palmitate",
    "Disodium EDTA",
    "Amodimethicone",
    "Sodium Citrate",
    "Palmitic Acid",
    "Sodium Hyaluronate",
    "T-Butyl Alcohol",
    "Tocopherol",
    "Octyldodecyl Myristate",
    "Denatonium Benzoate",
    "Citric Acid",
    "CI 42090 (Blue 1 Lake)",
    "CI 77510 (Ferric Ferrocyanide)"
]

ingredient_review.loc[
    mask, "corrected_ingredients"
] = ", ".join(ingredients)

ingredient_review.loc[
    mask, "verification_source"
] = "https://beiqonic.com/chanel-hydra-beauty-micro-creme-50-gr/"

ingredient_review.loc[
    mask, "review_decision"
] = "verified_retailer"

ingredient_review.loc[
    mask, "review_notes"
] = (
    "38 ingredients matched against a retailer listing "
    "for barcode 3145891410709. "
    "Corrected OCR and separator errors. "
    "Excluded formulation reference IL03A. "
    "This is the older formulation, not the newer "
    "HYDRA BEAUTY Micro Creme version."
)

display(
    ingredient_review.loc[
        mask,
        [
            "source_row",
            "product_name",
            "review_decision",
            "review_notes"
        ]
    ]
)

,source_row,product_name,review_decision,review_notes
3,95,Micro crème/ HYDRA BEAUTY,verified_retailer,38 ingredients matched against a retailer list...


In [25]:
ingredient_review.to_excel(
    interim_path / "manual_ingredient_review.xlsx",
    index=False
)

print("CHANEL row 95 saved successfully!")

CHANEL row 95 saved successfully!


In [26]:
# Select La Roche-Posay row 99
mask = ingredient_review["source_row"] == 99

ingredients = [
    "Aqua / Water",
    "Glycerin",
    "Cetearyl Alcohol",
    "Caprylic/Capric Triglyceride",
    "Glyceryl Stearate",
    "Alcohol Denat.",
    "Ethylhexyl Palmitate",
    "Dimethicone",
    "Butyrospermum Parkii Butter / Shea Butter",
    "Myristyl Myristate",
    "Niacinamide",
    "PEG-100 Stearate",
    "Glycine Soja Oil / Soybean Oil",
    "Sodium Polyacrylate",
    "Myristyl Malate Phosphonic Acid",
    "Disodium EDTA",
    "Caprylyl Glycol",
    "Citric Acid",
    "Tocopherol",
    "Pentaerythrityl Tetra-Di-T-Butyl Hydroxyhydrocinnamate",
    "Phenoxyethanol",
    "Parfum / Fragrance"
]

ingredient_review.loc[
    mask, "corrected_ingredients"
] = ", ".join(ingredients)

ingredient_review.loc[
    mask, "verification_source"
] = "https://www.laroche-posay.ca/en/nutritic-intense-riche/3337872413575.html"

ingredient_review.loc[
    mask, "review_decision"
] = "verified_manufacturer"

ingredient_review.loc[
    mask, "review_notes"
] = (
    "22 ingredient entries verified against the official "
    "La Roche-Posay website for barcode 3337872413575. "
    "Corrected separators and the OCR error "
    "'MYRISTYL O MYRISTATE'. "
    "Removed formulation reference B158565/1. "
    "An alternative formula exists in another source; "
    "this correction follows the manufacturer's listing."
)

display(
    ingredient_review.loc[
        mask,
        [
            "source_row",
            "product_name",
            "review_decision",
            "review_notes"
        ]
    ]
)

,source_row,product_name,review_decision,review_notes
4,99,Nutric Intense Rich In-depth Nutri-Reconstitut...,verified_manufacturer,22 ingredient entries verified against the off...


In [27]:
ingredient_review.to_excel(
    interim_path / "manual_ingredient_review.xlsx",
    index=False
)

print("La Roche-Posay row 99 saved successfully!")

La Roche-Posay row 99 saved successfully!


In [28]:
# Select L'Oréal row 108
mask = ingredient_review["source_row"] == 108

ingredients = [
    "Aqua / Water",
    "Glycerin",
    "Dimethicone",
    "Isohexadecane",
    "Hydroxyethylpiperazine Ethane Sulfonic Acid",
    "Alcohol Denat.",
    "Dipropylene Glycol",
    "Synthetic Wax",
    "Alpinia Galanga Leaf Extract",
    "PEG-10 Dimethicone",
    "PEG-30 Dipolyhydroxystearate",
    "Stearyl Alcohol",
    "Sorbitan Oleate",
    "Dimethicone/PEG-10/15 Crosspolymer",
    "Dimethicone/Polyglycerin-3 Crosspolymer",
    "Ceteareth-6",
    "Caffeine",
    "Sodium Acrylates Copolymer",
    "Sodium Citrate",
    "Sodium Hyaluronate",
    "Adenosine",
    "Nylon-12",
    "Disodium Acetyl Glucosamine Phosphate",
    "Disodium EDTA",
    "Isopropyl Alcohol",
    "Caprylic/Capric Triglyceride",
    "Caprylyl Glycol",
    "Xanthan Gum",
    "Pentylene Glycol",
    "Disteardimonium Hectorite",
    "Butylene Glycol",
    "Tocopherol",
    "Phenoxyethanol",
    "CI 60730 / Ext. Violet 2",
    "Linalool",
    "Citronellol",
    "Benzyl Alcohol",
    "Benzyl Benzoate",
    "Benzyl Salicylate",
    "Parfum / Fragrance"
]

ingredient_review.loc[
    mask, "corrected_ingredients"
] = ", ".join(ingredients)

ingredient_review.loc[
    mask, "verification_source"
] = (
    "https://drogerienatura.pl/produkt/"
    "loreal-paris-revitalift-filler-hialuronowe-"
    "wypelnienie-krem-antiage-na-noc-40-50-ml-"
    "3600523201501"
)

ingredient_review.loc[
    mask, "review_decision"
] = "verified_retailer"

ingredient_review.loc[
    mask, "review_notes"
] = (
    "40 ingredient entries transcribed from a retailer "
    "listing matching barcode 3600523201501. "
    "Original OCR text has missing separators and "
    "some differences from the retailer formulation. "
    "Manufacturer-label confirmation remains pending."
)

display(
    ingredient_review.loc[
        mask,
        [
            "source_row",
            "product_name",
            "review_decision",
            "review_notes"
        ]
    ]
)

,source_row,product_name,review_decision,review_notes
5,108,Revitalift filler,verified_retailer,40 ingredient entries transcribed from a retai...


In [29]:
print("Number of ingredients:", len(ingredients))

Number of ingredients: 40


In [30]:
ingredient_review.to_excel(
    interim_path / "manual_ingredient_review.xlsx",
    index=False
)

print("L'Oréal row 108 saved successfully!")

L'Oréal row 108 saved successfully!


In [31]:
# Select Jonzac row 114
mask = ingredient_review["source_row"] == 114

ingredients = [
    "Aqua / Water / Eau",
    "Hamamelis Virginiana (Witch Hazel) Leaf Water",
    "Malva Sylvestris (Mallow) Extract",
    "Glycerin",
    "Coco-Caprylate/Caprate",
    "Octyldodecanol",
    "Dicaprylyl Carbonate",
    "Lauryl Laurate",
    "Bentonite",
    "Butyrospermum Parkii (Shea) Butter",
    "Cetearyl Olivate",
    "Sorbitan Olivate",
    "Lactobacillus Ferment",
    "Lauroyl Lysine",
    "Xanthan Gum",
    "Sodium Levulinate",
    "Sclerotium Gum",
    "Bambusa Arundinacea Stem Extract",
    "Parfum (Fragrance)",
    "Zinc PCA",
    "Ricinus Communis (Castor) Seed Oil",
    "Lactic Acid",
    "Shikimic Acid",
    "Phytic Acid",
    "Lysolecithin",
    "Sodium Benzoate",
    "Hydrogenated Castor Oil",
    "Hydrolyzed Algin",
    "Zinc Sulfate",
    "Sodium Hydroxide",
    "Pullulan",
    "Oryza Sativa (Rice) Hull Powder",
    "Tocopherol",
    "Glycine Soja (Soybean) Oil"
]

ingredient_review.loc[
    mask, "corrected_ingredients"
] = ", ".join(ingredients)

ingredient_review.loc[
    mask, "verification_source"
] = (
    "https://www.pharmaciedesdrakkars.com/"
    "jonzac-pure-soin-purifiant-anti-imperfections"
)

ingredient_review.loc[
    mask, "review_decision"
] = "verified_retailer"

ingredient_review.loc[
    mask, "review_notes"
] = (
    "Ingredient list transcribed from a pharmacy listing "
    "matching barcode 3517360020793. "
    "Removed French explanatory descriptions, OCR artifacts "
    "and footnote numbers. "
    "Other retailer lists show minor formulation differences, "
    "so exact formulation-version confirmation remains pending."
)

display(
    ingredient_review.loc[
        mask,
        [
            "source_row",
            "product_name",
            "review_decision",
            "review_notes"
        ]
    ]
)

,source_row,product_name,review_decision,review_notes
6,114,Eau thermale Jonzac Pure soin purifiant anti-i...,verified_retailer,Ingredient list transcribed from a pharmacy li...


In [32]:
print("Number of ingredient entries:", len(ingredients))

Number of ingredient entries: 34


In [33]:
ingredient_review.to_excel(
    interim_path / "manual_ingredient_review.xlsx",
    index=False
)

print("Jonzac row 114 saved successfully!")

Jonzac row 114 saved successfully!


In [34]:
# Select Yves Rocher row 190
mask = ingredient_review["source_row"] == 190

ingredients = [
    "Aqua",
    "Glycerin",
    "Coco-Caprylate/Caprate",
    "Methylpropanediol",
    "Betaine",
    "Stearyl Alcohol",
    "Squalane",
    "Centaurea Cyanus Flower Water",
    "Butylene Glycol",
    "Arachidyl Alcohol",
    "Dimethicone",
    "Butyrospermum Parkii Butter",
    "Sorbitan Stearate",
    "Pyrus Malus Fruit Extract",
    "Aphloia Theiformis Leaf Extract",
    "Behenyl Alcohol",
    "Phenoxyethanol",
    "Parfum",
    "Arachidyl Glucoside",
    "Sodium Polyacrylate",
    "Retinyl Palmitate",
    "Ethyl Linoleate",
    "Caesalpinia Spinosa Gum",
    "Acrylates/C10-30 Alkyl Acrylate Crosspolymer",
    "Sorbic Acid",
    "Xanthan Gum",
    "Tocopheryl Acetate",
    "Tetrasodium EDTA",
    "Echinacea Angustifolia Meristem Cell Culture",
    "Maltodextrin",
    "Sodium Hydroxide",
    "Panax Ginseng Root Extract",
    "Sodium Benzoate",
    "Potassium Sorbate",
    "Silica",
    "Citric Acid",
    "Tocopherol",
    "Propyl Gallate"
]

ingredient_review.loc[
    mask, "corrected_ingredients"
] = ", ".join(ingredients)

ingredient_review.loc[
    mask, "verification_source"
] = (
    "https://www.codecheck.info/p/kosmetik/"
    "gesichtspflege/gesichtscremes/"
    "yves-rocher-anti-age-global-creme-fondamentale.html"
)

ingredient_review.loc[
    mask, "review_decision"
] = "verified_secondary_source"

ingredient_review.loc[
    mask, "review_notes"
] = (
    "Historical ingredient list closely matches "
    "the original dataset transcription. "
    "Product barcode 3660005148038 confirmed "
    "separately by Pilulka. "
    "Removed French explanatory text and "
    "formulation reference 1463v0. "
    "Exact barcode-to-formulation confirmation "
    "from the manufacturer remains unavailable."
)

display(
    ingredient_review.loc[
        mask,
        [
            "source_row",
            "product_name",
            "review_decision",
            "review_notes"
        ]
    ]
)

,source_row,product_name,review_decision,review_notes
7,190,Crème fondamentale Anti-âge global,verified_secondary_source,Historical ingredient list closely matches the...


In [35]:
print("Number of ingredient entries:", len(ingredients))

Number of ingredient entries: 38


In [36]:
ingredient_review.to_excel(
    interim_path / "manual_ingredient_review.xlsx",
    index=False
)

print("Yves Rocher row 190 saved successfully!")

Yves Rocher row 190 saved successfully!


In [37]:
# Show all 8 entries
display(
    ingredient_review[
        [
            "source_row",
            "brands",
            "review_decision"
        ]
    ]
)

print("\nReview status summary:")
print(
    ingredient_review["review_decision"]
    .value_counts(dropna=False)
)

,source_row,brands,review_decision
0,6,Stanhome,unavailable
1,9,Ceelini,verified_retailer
2,39,Garnier,verified_retailer
3,95,CHANEL,verified_retailer
4,99,La Roche-Posay,verified_manufacturer
5,108,L'Oréal,verified_retailer
6,114,Jonzac,verified_retailer
7,190,Yves Rocher,verified_secondary_source



Review status summary:
review_decision
verified_retailer            5
unavailable                  1
verified_manufacturer        1
verified_secondary_source    1
Name: count, dtype: int64


## Let's move to the additional single ingredient review 

In [38]:
single_review = pd.read_csv(
    interim_path / "additional_single_ingredient_review.csv",
    dtype={"code": "string"}
)

display(single_review)

,source_row,code,brands,product_name,ingredients_text,ingredient_review_status,review_decision,verification_source,review_notes
0,100,9310263000701,Mentholatum Aust. Pty Ltd,Oxy 5,Benzoyl Peroxide 5%,needs_manual_review,active_only,https://mentholatum.com.au/our-brands/oxy-range/,Official manufacturer confirms Benzoyl Peroxid...
1,146,3760354680187,Zenzitude,huile de jojoba vierge,Simmondsia Chinensis Seed Oil,needs_manual_review,plausible_single_ingredient,NaN,Dataset lists Simmondsia Chinensis Seed Oil as...
2,226,3284410034917,Melvita,Huile de jojoba,Simmondsia chinensis (jojoba) seed oil,needs_manual_review,formulation_unconfirmed,https://fr.melvita.com/products/huile-de-jojob...,Original dataset lists only Simmondsia Chinens...


In [39]:
# Row 100
mask = single_review["source_row"] == 100

single_review.loc[
    mask, "review_decision"
] = "active_only"

single_review.loc[
    mask, "verification_source"
] = "https://mentholatum.com.au/our-brands/oxy-range/"

single_review.loc[
    mask, "review_notes"
] = (
    "Official manufacturer confirms Benzoyl Peroxide 5% "
    "as the active ingredient. Complete inactive ingredient "
    "list not verified for this exact product. "
    "Classify as acne treatment, not moisturizer."
)

display(single_review.loc[mask])

,source_row,code,brands,product_name,ingredients_text,ingredient_review_status,review_decision,verification_source,review_notes
0,100,9310263000701,Mentholatum Aust. Pty Ltd,Oxy 5,Benzoyl Peroxide 5%,needs_manual_review,active_only,https://mentholatum.com.au/our-brands/oxy-range/,Official manufacturer confirms Benzoyl Peroxid...


In [40]:
# Category correction

if "category_review_decision" not in df_clean.columns:
    df_clean["category_review_decision"] = ""

df_clean.loc[
    df_clean["source_row"] == 100,
    "category_review_decision"
] = "reclassify_as_acne_treatment"

In [41]:
single_review.to_csv(
    interim_path / "additional_single_ingredient_review.csv",
    index=False
)

print("Row 100 review saved successfully!")

Row 100 review saved successfully!


In [42]:
# Select Zenzitude row 146
mask = single_review["source_row"] == 146

single_review.loc[
    mask, "review_decision"
] = "plausible_single_ingredient"

single_review.loc[
    mask, "verification_source"
] = ""

single_review.loc[
    mask, "review_notes"
] = (
    "Dataset lists Simmondsia Chinensis Seed Oil "
    "as the only ingredient. Pure jojoba oil can "
    "legitimately contain one INCI ingredient. "
    "No parsing error identified. "
    "Exact barcode-level formulation confirmation "
    "not available. Manufacturer-label verification "
    "remains pending."
)

display(
    single_review.loc[
        mask,
        [
            "source_row",
            "product_name",
            "review_decision",
            "review_notes"
        ]
    ]
)

,source_row,product_name,review_decision,review_notes
1,146,huile de jojoba vierge,plausible_single_ingredient,Dataset lists Simmondsia Chinensis Seed Oil as...


In [43]:
if "category_review_decision" not in df_clean.columns:
    df_clean["category_review_decision"] = ""

df_clean.loc[
    df_clean["source_row"] == 146,
    "category_review_decision"
] = "reclassify_as_face_body_oil"

display(
    df_clean.loc[
        df_clean["source_row"] == 146,
        [
            "source_row",
            "product_name",
            "role",
            "category_review_decision"
        ]
    ]
)

,source_row,product_name,role,category_review_decision
146,146,huile de jojoba vierge,moisturizer,reclassify_as_face_body_oil


In [44]:
single_review.to_csv(
    interim_path / "additional_single_ingredient_review.csv",
    index=False
)

print("Zenzitude row 146 saved successfully!")

Zenzitude row 146 saved successfully!


In [45]:
# Select Melvita row 226
mask = single_review["source_row"] == 226

single_review.loc[
    mask, "review_decision"
] = "formulation_unconfirmed"

single_review.loc[
    mask, "verification_source"
] = "https://fr.melvita.com/products/huile-de-jojoba-bio"

single_review.loc[
    mask, "review_notes"
] = (
    "Original dataset lists only Simmondsia Chinensis "
    "(Jojoba) Seed Oil. A single-ingredient formulation "
    "is documented for Melvita jojoba oil, but another "
    "Melvita formulation contains Tocopherol and "
    "Glycine Soja Oil. Exact formulation for barcode "
    "3284410034917 could not be confirmed. "
    "Original ingredient list retained."
)

display(
    single_review.loc[
        mask,
        [
            "source_row",
            "product_name",
            "review_decision",
            "review_notes"
        ]
    ]
)

,source_row,product_name,review_decision,review_notes
2,226,Huile de jojoba,formulation_unconfirmed,Original dataset lists only Simmondsia Chinens...


In [46]:
if "category_review_decision" not in df_clean.columns:
    df_clean["category_review_decision"] = ""

df_clean.loc[
    df_clean["source_row"] == 226,
    "category_review_decision"
] = "reclassify_as_face_body_oil"

display(
    df_clean.loc[
        df_clean["source_row"] == 226,
        [
            "source_row",
            "product_name",
            "role",
            "category_review_decision"
        ]
    ]
)

,source_row,product_name,role,category_review_decision
226,226,Huile de jojoba,moisturizer,reclassify_as_face_body_oil


In [47]:
single_review.to_csv(
    interim_path / "additional_single_ingredient_review.csv",
    index=False
)

print("Melvita row 226 saved successfully!")

Melvita row 226 saved successfully!


In [48]:
display(
    single_review[
        [
            "source_row",
            "brands",
            "product_name",
            "review_decision"
        ]
    ]
)

,source_row,brands,product_name,review_decision
0,100,Mentholatum Aust. Pty Ltd,Oxy 5,active_only
1,146,Zenzitude,huile de jojoba vierge,plausible_single_ingredient
2,226,Melvita,Huile de jojoba,formulation_unconfirmed


## Reviewing the duplicates 

In [49]:
# Load your duplicate review file
duplicate_review = pd.read_csv(
    interim_path / "duplicate_manual_review.csv",
    dtype={"code_1": "string", "code_2": "string"}
)

In [50]:
# Rows 175 and 176

mask = (
    (duplicate_review["row_1"] == 175)
    & (duplicate_review["row_2"] == 176)
)

# Record our decision
duplicate_review.loc[
    mask, "review_decision"
] = "keep_separate"

duplicate_review.loc[
    mask, "review_notes"
] = (
    "Both records have identical product names but "
    "different barcodes and ingredient differences. "
    "Row 175 contains Xanthan Gum, Caprylyl Glycol "
    "and Ethylhexylglycerin. Row 176 contains "
    "Sodium Polyacrylate and P-Anisic Acid. "
    "The split Sodium Stearoyl/Lactylate entry "
    "in row 176 requires correction. "
    "Likely formulation variants; retain both."
)

duplicate_review.loc[
    mask, "verification_source"
] = (
    "https://incidecoder.com/products/"
    "dm-babylove-babylove-leichte-gesichtcreme"
)

display(
    duplicate_review.loc[
        mask,
        [
            "row_1",
            "row_2",
            "brand",
            "review_decision",
            "review_notes"
        ]
    ]
)


,row_1,row_2,brand,review_decision,review_notes
0,175,176,babylove,keep_separate,Both records have identical product names but ...


In [51]:
duplicate_review.to_csv(
    interim_path / "duplicate_manual_review.csv",
    index=False
)

print("Babylove duplicate review saved!")

Babylove duplicate review saved!


In [52]:
# Rows 49 and 56
def normalize_for_comparison(text):
    return {
        ingredient.strip().lower().rstrip(" .;")
        for ingredient in str(text).split(",")
        if ingredient.strip()
    }

ing49 = normalize_for_comparison(
    df_clean.loc[
        df_clean["source_row"] == 49,
        "ingredients_text"
    ].iloc[0]
)

ing56 = normalize_for_comparison(
    df_clean.loc[
        df_clean["source_row"] == 56,
        "ingredients_text"
    ].iloc[0]
)

similarity = (
    len(ing49 & ing56) / len(ing49 | ing56) * 100
)

print("Ingredient similarity:", round(similarity, 2), "%")
print("Same ingredient sets:", ing49 == ing56)

Ingredient similarity: 100.0 %
Same ingredient sets: True


In [53]:
# Select Bioderma Atoderm pair
mask = (
    (duplicate_review["row_1"] == 49)
    & (duplicate_review["row_2"] == 56)
)

# Record the decision
duplicate_review.loc[
    mask, "review_decision"
] = "keep_separate"

# Explain the decision
duplicate_review.loc[
    mask, "review_notes"
] = (
    "Same Bioderma Atoderm Cream Ultra product family. "
    "Row 49 barcode 8699956514185 is 200 ml; "
    "row 56 barcode 8699956513676 is 500 ml. "
    "Ingredient lists match 100% after removing "
    "trailing punctuation. Different packaging sizes "
    "have distinct barcodes, so both records are retained."
)

# Document verification sources
duplicate_review.loc[
    mask, "verification_source"
] = (
    "https://www.ahona.store/products/"
    "bioderma-atoderm-cream-ultra-514185; "
    "https://doktorpharma.com/"
    "bioderma-atoderm-cream-ultra-500-ml-puansiz-1702"
)

display(
    duplicate_review.loc[
        mask,
        [
            "row_1",
            "row_2",
            "brand",
            "review_decision",
            "review_notes"
        ]
    ]
)

,row_1,row_2,brand,review_decision,review_notes
1,49,56,bioderma,keep_separate,Same Bioderma Atoderm Cream Ultra product fami...


In [54]:
duplicate_review.to_csv(
    interim_path / "duplicate_manual_review.csv",
    index=False
)

print("Bioderma Atoderm review saved successfully!")

Bioderma Atoderm review saved successfully!


In [55]:
# Rows 85 and 136 Bioderma Sébium Sensitive

mask = (
    (duplicate_review["row_1"] == 85)
    & (duplicate_review["row_2"] == 136)
)

duplicate_review.loc[
    mask, "review_decision"
] = "needs_review"

duplicate_review.loc[
    mask, "review_notes"
] = (
    "Both records have identical names and all 25 "
    "ingredients in the same order. "
    "Row 85 has valid EAN-13 barcode 3401360106994. "
    "Row 136 has short code 06010699, which is not "
    "a valid EAN-8 barcode. A secondary ingredient "
    "database associates both codes with the same "
    "30 ml product. Probable duplicate; verify "
    "identifier before merging."
)

duplicate_review.loc[
    mask, "verification_source"
] = (
    "https://www.pharmalite.com/prodotto/"
    "sebium-sensitive-30ml/ ; "
    "https://www.indecoder.ru/es/product/"
    "bioderma-s-bium-sensitive-soothing-anti-blemish-care"
)

display(
    duplicate_review.loc[
        mask,
        [
            "row_1",
            "row_2",
            "review_decision",
            "review_notes"
        ]
    ]
)

,row_1,row_2,review_decision,review_notes
2,85,136,needs_review,Both records have identical names and all 25 i...


In [56]:
duplicate_review.to_csv(
    interim_path / "duplicate_manual_review.csv",
    index=False
)

print("Bioderma Sébium review saved!")


Bioderma Sébium review saved!


In [57]:
# Select rows 78 and 191 Kiehl's duplicate pair

mask = (
    (duplicate_review["row_1"] == 78)
    & (duplicate_review["row_2"] == 191)
)

# Record the decision
duplicate_review.loc[
    mask, "review_decision"
] = "keep_separate"

# Explain the decision
duplicate_review.loc[
    mask, "review_notes"
] = (
    "Both records belong to the Kiehl's Ultra Facial "
    "Cream product family, but have different barcodes "
    "and substantially different ingredient lists. "
    "Row 78 contains Dimethicone, Ethylhexylglycerin "
    "and Trisodium Ethylenediamine Disuccinate. "
    "Row 191 contains Cyclohexasiloxane, Urea, "
    "Methylparaben, Propylparaben and Disodium EDTA. "
    "The original ingredient similarity is 35.29%, "
    "partly affected by spelling and formatting. "
    "Likely different formulation versions. "
    "Retain both records. Exact barcode-to-formulation "
    "confirmation for row 191 remains pending."
)

# Document supporting sources
duplicate_review.loc[
    mask, "verification_source"
] = (
    "https://www.kiehls.com/Ultra-Facial-Cream/"
    "3605970360757.html; "
    "https://nationaleczema.org/eczema-products/"
    "ultra-facial-cream/"
)

display(
    duplicate_review.loc[
        mask,
        [
            "row_1",
            "row_2",
            "brand",
            "review_decision",
            "review_notes"
        ]
    ]
)

,row_1,row_2,brand,review_decision,review_notes
3,78,191,kiehl's,keep_separate,Both records belong to the Kiehl's Ultra Facia...


In [58]:
duplicate_review.to_csv(
    interim_path / "duplicate_manual_review.csv",
    index=False
)

print("Kiehl's duplicate review saved successfully!")


Kiehl's duplicate review saved successfully!


In [59]:
# Rows 108 and 186

mask = (
    (duplicate_review["row_1"] == 108)
    & (duplicate_review["row_2"] == 186)
)

print("Matching pairs:", mask.sum())

Matching pairs: 1


In [60]:
duplicate_review.loc[
    mask, "review_decision"
] = "keep_separate"

duplicate_review.loc[
    mask, "review_notes"
] = (
    "Barcode 3600523201501 identifies L'Oreal Paris "
    "Revitalift Filler Night Cream 50 ml. "
    "Barcode 3600523732548 identifies Revitalift "
    "Filler seven-day plumping ampoules. "
    "These are distinct product types, not duplicates. "
    "The original 0% ingredient similarity is invalid "
    "because row 108 had missing ingredient separators. "
    "Retain both products and reclassify row 186 "
    "as a serum/treatment."
)

duplicate_review.loc[
    mask, "verification_source"
] = (
    "https://superpharm.it/loreal-paris-revitalift-filler-crema-notte-50-ml-116131; "
    "https://frenchbeautyhub.com/en-ae/products/"
    "lor-al-paris-revitalift-filler-7-day-course-of-treatment-"
    "plumping-ampulas-1-3ml-x7-0-04fl-oz-x7"
)

display(
    duplicate_review.loc[
        mask,
        ["row_1", "row_2", "review_decision", "review_notes"]
    ]
)

,row_1,row_2,review_decision,review_notes
4,108,186,keep_separate,Barcode 3600523201501 identifies L'Oreal Paris...


In [61]:
# Flag row 186 for category correction

if "category_review_decision" not in df_clean.columns:
    df_clean["category_review_decision"] = ""

df_clean.loc[
    df_clean["source_row"] == 186,
    "category_review_decision"
] = "reclassify_as_serum_treatment"

display(
    df_clean.loc[
        df_clean["source_row"].isin([108, 186]),
        [
            "source_row",
            "product_name",
            "role",
            "category_review_decision"
        ]
    ]
)

,source_row,product_name,role,category_review_decision
108,108,Revitalift filler,moisturizer,
186,186,Revitalift filler,moisturizer,reclassify_as_serum_treatment


In [62]:
duplicate_review.to_csv(
    interim_path / "duplicate_manual_review.csv",
    index=False
)

print("L'Oréal duplicate review saved successfully!")

L'Oréal duplicate review saved successfully!


In [63]:
# Rows 32 and 84

nivea_rows = df_clean[
    df_clean["source_row"].isin([32, 84])
]

display(
    nivea_rows[
        ["source_row", "code", "brands",
         "product_name", "ingredients_text"]
    ]
)

,source_row,code,brands,product_name,ingredients_text
32,32,7501054500193,nivea,Nivea creme,"Aqua, Paraffinum Liquidum, Cera Microcristalli..."
84,84,4005800001192,Nivea,Nivea Creme,"Aqua, Paraffinum Liquidum, Cera Microcristalli..."


In [64]:
# Compare ingredients

def parse_ingredients(text):
    return {
        ingredient.strip().lower().rstrip(" .;")
        for ingredient in str(text).split(",")
        if ingredient.strip()
    }

row32 = df_clean.loc[
    df_clean["source_row"] == 32,
    "ingredients_text"
].iloc[0]

row84 = df_clean.loc[
    df_clean["source_row"] == 84,
    "ingredients_text"
].iloc[0]

ing32 = parse_ingredients(row32)
ing84 = parse_ingredients(row84)

print("ROW 32 INGREDIENT COUNT:", len(ing32))
print("ROW 84 INGREDIENT COUNT:", len(ing84))

print("\nONLY IN ROW 32:")
for ingredient in sorted(ing32 - ing84):
    print("-", ingredient)

print("\nONLY IN ROW 84:")
for ingredient in sorted(ing84 - ing32):
    print("-", ingredient)

print("\nCOMMON INGREDIENTS:")
for ingredient in sorted(ing32 & ing84):
    print("-", ingredient)

similarity = len(ing32 & ing84) / len(ing32 | ing84) * 100

print(f"\nNormalized similarity: {similarity:.2f}%")

ROW 32 INGREDIENT COUNT: 15
ROW 84 INGREDIENT COUNT: 21

ONLY IN ROW 32:
- lanolin alcohol (eucerit®)
- sodium anisate

ONLY IN ROW 84:
- benzyl benzoate
- cinnamyl alcohol
- citronellol
- geraniol
- hydroxycitronellal
- lanolin alcohol (eucerit)
- limonene
- linalool

COMMON INGREDIENTS:
- aluminum stearates
- aqua
- cera microcristallina
- citric acid
- decyl oleate
- glycerin
- magnesium stearate
- magnesium sulfate
- octyldodecanol
- panthenol
- paraffin
- paraffinum liquidum
- parfum

Normalized similarity: 56.52%


In [65]:
# Select Nivea pair
mask = (
    ((duplicate_review["row_1"] == 32) &
     (duplicate_review["row_2"] == 84))
    |
    ((duplicate_review["row_1"] == 84) &
     (duplicate_review["row_2"] == 32))
)

duplicate_review.loc[
    mask, "review_decision"
] = "keep_separate"

duplicate_review.loc[
    mask, "review_notes"
] = (
    "Row 32 barcode 7501054500193 identifies "
    "Nivea Creme 200 ml sold in Mexico. "
    "Row 84 barcode 4005800001192 identifies "
    "Nivea Creme 150 ml sold in Germany. "
    "Official Mexican and German retailer sources "
    "publish differing ingredient declarations. "
    "Exact differences in the dataset's original "
    "ingredient lists remain to be checked. "
    "Distinct retail products; retain both records."
)

duplicate_review.loc[
    mask, "verification_source"
] = (
    "https://www.nivea.com.mx/productos/"
    "nivea-creme-75010545001930060.html; "
    "https://www.shop-apotheke.com/beauty/"
    "727133/nivea-creme.htm"
)

# Save
duplicate_review.to_csv(
    interim_path / "duplicate_manual_review.csv",
    index=False
)

print("Nivea duplicate decision updated!")

Nivea duplicate decision updated!


In [66]:
duplicate_review.loc[
    mask, "review_decision"
] = "needs_review"

duplicate_review.loc[
    mask, "review_notes"
] = (
    "Both records are Nivea Creme, but they have "
    "different product codes and an original "
    "ingredient-set similarity of 56.52%. "
    "Manufacturer websites document different "
    "ingredient declarations across markets. "
    "Exact barcode-to-formulation mapping has not "
    "been verified. Retain both provisionally; "
    "do not merge without checking both identities."
)

duplicate_review.loc[
    mask, "verification_source"
] = (
    "https://www.niveausa.com/products/"
    "nivea-creme-721400054670079.html; "
    "https://www.nivea.in/products/"
    "nivea-creme-422042680213.html"
)

display(
    duplicate_review.loc[
        mask,
        ["row_1", "row_2", "review_decision", "review_notes"]
    ]
)

,row_1,row_2,review_decision,review_notes
5,32,84,needs_review,"Both records are Nivea Creme, but they have di..."


In [67]:
duplicate_review.to_csv(
    interim_path / "duplicate_manual_review.csv",
    index=False
)

print("Nivea duplicate review saved successfully!")

Nivea duplicate review saved successfully!


In [68]:
# Rows 0 and 1

mask = (
    ((duplicate_review["row_1"] == 0) &
     (duplicate_review["row_2"] == 1))
    |
    ((duplicate_review["row_1"] == 1) &
     (duplicate_review["row_2"] == 0))
)

print("Matching pairs:", mask.sum())

assert mask.sum() == 1, "Expected exactly one PLU pair."

Matching pairs: 1


In [69]:
duplicate_review.loc[
    mask, "review_decision"
] = "needs_review"

duplicate_review.loc[
    mask, "review_notes"
] = (
    "Both records represent PLU Laboratories GHK-Cu "
    "Face Tonic with identical six-ingredient formulas. "
    "Row 0 barcode: 0860015354624. "
    "Row 1 barcode: 0860015354648. "
    "The manufacturer confirms 10 mL and 30 mL "
    "sizes, but neither barcode could be reliably "
    "mapped to a specific size. "
    "Retain both provisionally until packaging "
    "or barcode identity is verified."
)

duplicate_review.loc[
    mask, "verification_source"
] = (
    "https://plulaboratories.com/"
    "products/ghk-cu-face-tonic"
)

display(
    duplicate_review.loc[
        mask,
        ["row_1", "row_2", "review_decision", "review_notes"]
    ]
)

,row_1,row_2,review_decision,review_notes
6,0,1,needs_review,Both records represent PLU Laboratories GHK-Cu...


In [70]:
# Correct the product categories

if "category_review_decision" not in df_clean.columns:
    df_clean["category_review_decision"] = ""

df_clean.loc[
    df_clean["source_row"].isin([0, 1]),
    "category_review_decision"
] = "reclassify_as_serum_treatment"

display(
    df_clean.loc[
        df_clean["source_row"].isin([0, 1]),
        [
            "source_row",
            "product_name",
            "role",
            "category_review_decision"
        ]
    ]
)

,source_row,product_name,role,category_review_decision
0,0,PLU Laboratories GHK-Cu Face Tonic,moisturizer,reclassify_as_serum_treatment
1,1,PLU Laboratories GHK-Cu Face Tonic,moisturizer,reclassify_as_serum_treatment


In [71]:
duplicate_review.to_csv(
    interim_path / "duplicate_manual_review.csv",
    index=False
)

print("PLU Laboratories review saved successfully!")

PLU Laboratories review saved successfully!


In [72]:
# Rows 62 and 63

# Display both products and their exact barcodes
pair = df_clean.loc[
    df_clean["source_row"].isin([62, 63]),
    ["source_row", "code", "brands",
     "product_name", "ingredients_text"]
]

for _, row in pair.iterrows():
    print(f"\nROW {row['source_row']}")
    print("Barcode:", row["code"])
    print("Product:", row["product_name"])
    print("Ingredients:", row["ingredients_text"])


ROW 62
Barcode: 5060879821989
Product: Vitamin B, C and E moisturizer
Ingredients: Water, Glycerin, Glyceryl Stearate SE, Cetearyl Alcohol, Caprylic/Capric Triglyceride, Cocos Nucifera (Coconut) Oil, Stearic Acid, Phenoxyethanol, Benzyl Alcohol, Carbomer, Sodium Hydroxide, Niacinamide, Panthenol, Sodium Ascorbyl Phosphate, Sodium Hyaluronate, Tocopheryl Acetate, Ethylhexylglycerin, Dehydroacetic Acid, Lecithin, Trisodium Ethylenediamine Disuccinate, Ascorbyl Palmitate, Tocopherol, Helianthus Annuus (Sunflower) Seed Oil

ROW 63
Barcode: 5060422297346
Product: Vitamin B, C and E moisturizer
Ingredients: Water, Glycerin, Glyceryl Stearate SE, Cetearyl Alcohol, Caprylic/Capric Triglyceride, Cocos Nucifera 
(Coconut) Oil, Stearic Acid, Phenoxyethanol, Benzyl Alcohol, Carbomer, Sodium Hydroxide, Niacinamide, Panthenol, Sodium Ascorbyl Phosphate, Sodium Hyaluronate, Tocopheryl Acetate, Ethylhexylglycerin, Dehydroacetic Acid, Lecithin, Trisodium Ethylenediamine Disuccinate, Ascorbyl Palmitate

In [73]:

mask = (
    ((duplicate_review["row_1"] == 62) &
     (duplicate_review["row_2"] == 63))
    |
    ((duplicate_review["row_1"] == 63) &
     (duplicate_review["row_2"] == 62))
)

assert mask.sum() == 1

duplicate_review.loc[
    mask, "review_decision"
] = "merge_as_duplicate"

duplicate_review.loc[
    mask, "review_notes"
] = (
    "Both records identify The INKEY List Vitamin B, C "
    "and E Moisturizer 50ml. Row 62 barcode "
    "5060879821989 and row 63 barcode 5060422297346. "
    "All 23 ingredients match in the same order after "
    "whitespace normalization. Original similarity "
    "91.67% was affected by formatting. "
    "Recommend one catalogue product while preserving "
    "both source rows and barcodes in the audit trail."
)

duplicate_review.loc[
    mask, "verification_source"
] = (
    "https://www.dischem.co.za/the-inkey-list-vitamin-b-c-and-e-moisturizer-50ml-585; "
    "https://www.onbuy.com/gb/p/the-inkey-list-vitamin-b-c-and-e-lightweight-daily-face-moisturiser-to-keep-skin-healthy-50ml~p56055579/"
)


In [74]:
duplicate_review.to_csv(
    interim_path / "duplicate_manual_review.csv",
    index=False
)

print("Final duplicate decision saved!")

Final duplicate decision saved!
